# Data Preprocessing
## Gym Members Exercise Dataset

**Author:** Sarah Suliman  
**Course:** CSPB 4502: Data Mining  
**Project:** Discovering Fitness Profiles and Workout Performance Patterns Through Data Mining

## 1. Objectives

The goal of this notebook is to prepare the Gym Members Exercise Dataset for machine learning analyses. Data preprocessing improves data quality, validates the dataset, and creates additional features that will support clustering, classification, and regression models.

The preprocessing workflow includes:

- Loading the dataset
- Checking for missing values
- Detecting duplicate records
- Validating numerical ranges
- Validating categorical variables
- Creating derived features
- Saving the processed dataset

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
DATA_PATH = Path("../data/gym_members_exercise_tracking.csv")
OUTPUT_DIR = Path("../data/processed")

OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)

df.head()

,Age,Gender,Weight (kg),Height (m),Max_BPM,Avg_BPM,Resting_BPM,Session_Duration (hours),Calories_Burned,Workout_Type,Fat_Percentage,Water_Intake (liters),Workout_Frequency (days/week),Experience_Level,BMI
0,56,Male,88.3,1.71,180,157,60,1.69,1313.0,Yoga,12.6,3.5,4,3,30.20
1,46,Female,74.9,1.53,179,151,66,1.30,883.0,HIIT,33.9,2.1,4,2,32.00
2,32,Female,68.1,1.66,167,122,54,1.11,677.0,Cardio,33.4,2.3,4,2,24.71
3,25,Male,53.2,1.70,190,164,56,0.59,532.0,Strength,28.8,2.1,3,1,18.41
4,38,Male,46.1,1.79,188,158,68,0.64,556.0,Strength,29.2,2.8,3,1,14.39


## 2. Initial Dataset Inspection

In [3]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

Rows: 973
Columns: 15


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 973 entries, 0 to 972
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Age                            973 non-null    int64  
 1   Gender                         973 non-null    object 
 2   Weight (kg)                    973 non-null    float64
 3   Height (m)                     973 non-null    float64
 4   Max_BPM                        973 non-null    int64  
 5   Avg_BPM                        973 non-null    int64  
 6   Resting_BPM                    973 non-null    int64  
 7   Session_Duration (hours)       973 non-null    float64
 8   Calories_Burned                973 non-null    float64
 9   Workout_Type                   973 non-null    object 
 10  Fat_Percentage                 973 non-null    float64
 11  Water_Intake (liters)          973 non-null    float64
 12  Workout_Frequency (days/week)  973 non-null    int

The dataset contains 973 observations and 15 variables. All columns contain complete data, providing a strong foundation for preprocessing and machine learning.

## 3. Missing Value Validation

In [5]:
missing_values = df.isnull().sum()

missing_values

Age                              0
Gender                           0
Weight (kg)                      0
Height (m)                       0
Max_BPM                          0
Avg_BPM                          0
Resting_BPM                      0
Session_Duration (hours)         0
Calories_Burned                  0
Workout_Type                     0
Fat_Percentage                   0
Water_Intake (liters)            0
Workout_Frequency (days/week)    0
Experience_Level                 0
BMI                              0
dtype: int64

No missing values were detected. Because the dataset is complete, no imputation techniques were required before continuing with the analysis.

## 4. Duplicate Detection

In [6]:
duplicate_count = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_count}")

Duplicate rows: 0


No duplicate observations were identified. Therefore, no records were removed during preprocessing.

## 5. Numerical Range Validation

In [7]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,973.0,38.683453,12.180928,18.00,28.00,40.00,49.00,59.00
Weight (kg),973.0,73.854676,21.207500,40.00,58.10,70.00,86.00,129.90
Height (m),973.0,1.722580,0.127720,1.50,1.62,1.71,1.80,2.00
Max_BPM,973.0,179.883864,11.525686,160.00,170.00,180.00,190.00,199.00
Avg_BPM,973.0,143.766701,14.345101,120.00,131.00,143.00,156.00,169.00
Resting_BPM,973.0,62.223022,7.327060,50.00,56.00,62.00,68.00,74.00
Session_Duration (hours),973.0,1.256423,0.343033,0.50,1.04,1.26,1.46,2.00
Calories_Burned,973.0,905.422405,272.641516,303.00,720.00,893.00,1076.00,1783.00
Fat_Percentage,973.0,24.976773,6.259419,10.00,21.30,26.20,29.30,35.00
Water_Intake (liters),973.0,2.626619,0.600172,1.50,2.20,2.60,3.10,3.70


The numerical variables fall within realistic ranges for human physiological and workout measurements. No impossible or invalid values were identified.

## 6. Category Validation

In [8]:
print(df["Gender"].unique())

print(df["Workout_Type"].unique())

print(sorted(df["Experience_Level"].unique()))

['Male' 'Female']
['Yoga' 'HIIT' 'Cardio' 'Strength']
[np.int64(1), np.int64(2), np.int64(3)]


The categorical variables contain valid values and require no additional cleaning before modeling.

## 7. Feature Engineering

In [9]:
processed_df = df.copy()

In [10]:
processed_df["Heart_Rate_Reserve"] = (
    processed_df["Max_BPM"] -
    processed_df["Resting_BPM"]
)

In [11]:
processed_df["Calories_Per_Hour"] = (
    processed_df["Calories_Burned"] /
    processed_df["Session_Duration (hours)"]
)

In [12]:
processed_df["Water_Intake_Per_Hour"] = (
    processed_df["Water_Intake (liters)"] /
    processed_df["Session_Duration (hours)"]
)

In [13]:
processed_df["BMI_Category"] = pd.cut(
    processed_df["BMI"],
    bins=[0,18.5,25,30,np.inf],
    labels=[
        "Underweight",
        "Normal",
        "Overweight",
        "Obese"
    ]
)

Four additional variables were engineered to provide more meaningful measures of exercise intensity and body composition. These derived features will be incorporated into the clustering, classification, and regression analyses.

## 8. Processed Dataset Preview

In [14]:
processed_df.head()

,Age,Gender,Weight (kg),Height (m),Max_BPM,Avg_BPM,Resting_BPM,Session_Duration (hours),Calories_Burned,Workout_Type,Fat_Percentage,Water_Intake (liters),Workout_Frequency (days/week),Experience_Level,BMI,Heart_Rate_Reserve,Calories_Per_Hour,Water_Intake_Per_Hour,BMI_Category
0,56,Male,88.3,1.71,180,157,60,1.69,1313.0,Yoga,12.6,3.5,4,3,30.20,120,776.923077,2.071006,Obese
1,46,Female,74.9,1.53,179,151,66,1.30,883.0,HIIT,33.9,2.1,4,2,32.00,113,679.230769,1.615385,Obese
2,32,Female,68.1,1.66,167,122,54,1.11,677.0,Cardio,33.4,2.3,4,2,24.71,113,609.909910,2.072072,Normal
3,25,Male,53.2,1.70,190,164,56,0.59,532.0,Strength,28.8,2.1,3,1,18.41,134,901.694915,3.559322,Underweight
4,38,Male,46.1,1.79,188,158,68,0.64,556.0,Strength,29.2,2.8,3,1,14.39,120,868.750000,4.375000,Underweight


In [15]:
processed_df.shape

(973, 19)

The processed dataset now contains four engineered variables, increasing the total number of attributes from 15 to 19 while maintaining all original observations.

## 9. Save Processed Dataset

In [16]:
output_path = OUTPUT_DIR / "gym_members_exercise_cleaned.csv"

processed_df.to_csv(
    output_path,
    index=False
)

print(f"Processed dataset saved to:\n{output_path}")

Processed dataset saved to:
../data/processed/gym_members_exercise_cleaned.csv


## 10. Summary

The preprocessing workflow confirmed that the dataset contained no missing values or duplicate records and that all numerical and categorical variables were valid. Four engineered features were added to improve the representation of workout intensity and body composition. The cleaned dataset now contains 973 observations and 19 variables and is ready for clustering, classification, and regression analyses.